# GPU SHA-256 — Validation Walkthrough

Runs the full project pipeline on Colab: generate dataset → smoke test → validate GPU vs CPU.

> **Before you start:** Menu → **Runtime → Change runtime type → T4 GPU → Save**, then reconnect.  
> If you skip this, `nvcc` will not be found.

## Step 1 — Environment check

Run this first. It confirms the GPU is attached and adds `nvcc` to the PATH.  
If `nvidia-smi` errors, go to **Runtime → Change runtime type → GPU** and reconnect.

In [ ]:
import os
import subprocess

# Add CUDA bin to PATH — fixes 'nvcc: command not found' on some Colab instances
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')

# Confirm GPU is attached
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → GPU and reconnect.')
print('GPU:', result.stdout.strip())

# Confirm nvcc is reachable
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('nvcc not found even after PATH fix. Try Runtime → Factory reset runtime.')
print('nvcc:', result.stdout.splitlines()[3])
print('\nENVIRONMENT OK')

## Step 2 — Clone repo

In [ ]:
import os

if not os.path.exists('gpu_assignment'):
    !git clone https://github.com/ananpal/gpu_assignment.git

%cd gpu_assignment
!git fetch origin
!git checkout kernel-gpu-loader
!git pull origin kernel-gpu-loader
!ls

## Step 3 — Install OpenSSL headers

Needed to compile the validator against OpenSSL SHA-256.

In [ ]:
!apt-get install -y libssl-dev 2>&1 | tail -3
!pkg-config --modversion libssl && echo 'OpenSSL OK'

## Step 4 — Check dataset

The repo may already contain a committed dataset in `data/`.  
This cell inspects it — if all required files are present you can **skip Step 5** and go straight to the smoke test.

In [ ]:
import os

required = [
    'data/meta.txt',
    'data/messages.bin',
    'data/offsets.bin',
    'data/lengths.bin',
    'data/expected_digests.bin',
]

print('=== Dataset in repo ===')
all_present = True
for f in required:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f):,} bytes' if exists else 'MISSING'
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {f:<35} {size}')
    if not exists:
        all_present = False

if all_present:
    with open('data/meta.txt') as f:
        meta = f.read().strip()
    num = int(meta.split('=')[1])
    print(f'\n  {meta}  ({num:,} messages)')
    print('\nDataset READY — you can skip Step 5 and go straight to Step 6 (smoke test).')
else:
    print('\nDataset incomplete — run Step 5 to generate it.')

## Step 5 — Generate dataset *(skip if Step 4 shows READY)*

Only needed if the dataset files are missing or you want a fresh run at a different size.

In [ ]:
!python src/cpu_reference/generate_dataset.py 10000
!echo '--- data/ ---' && ls -lh data/

## Step 6 — Smoke test

Hashes 4 NIST vectors (empty string, `"abc"`, `"hello"`, 56-byte string) on the GPU.  
**Must print ALL PASS before running the validator.** A FAIL here means a bug in the SHA-256 math.

In [ ]:
!nvcc src/kernel/sha256_smoke_test.cu -o sha256_smoke_test
!./sha256_smoke_test
# expect: ALL PASS

## Step 7 — Compile the validator

Links `validate.cpp` (Arundhati) against the GPU engine `sha256_gpu.cu` (Anand).

In [ ]:
!nvcc src/validate/validate.cpp src/kernel/sha256_gpu.cu \
      -o validate -lssl -lcrypto
print('compiled OK')

## Step 8 — Run the validator

Two checks in one run:
1. **Edge-case suite** — empty, 1 byte, 55/56/64-byte boundaries, multi-block, NIST vectors.
2. **Dataset check** — all messages on GPU vs CPU reference, first mismatch reported if any.

In [ ]:
!./validate data
# expect:
#   == Edge-case suite ==
#   [PASS] empty(0)  [PASS] 1 byte  ... edge cases: ALL PASS
#   == Dataset check (data, N messages) ==
#   ALL MATCH (N messages)
#   VALIDATION PASSED

## Step 9 — Scale up

Change `N` and re-run. A dedicated GPU machine handles 10M comfortably; Colab T4 is fine up to ~5M.

In [ ]:
N = 1_000_000  # change to 10_000_000 for the final benchmark run

!python src/cpu_reference/generate_dataset.py {N}
!./validate data

---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `nvcc: command not found` | Re-run Step 1 — it adds CUDA to PATH. If that fails, go to **Runtime → Factory reset runtime** |
| `No GPU found` | **Runtime → Change runtime type → T4 GPU → Save**, then reconnect and start from Step 1 |
| Smoke test `FAIL` | SHA-256 math bug in `include/sha256.cuh` — do not proceed to the validator |
| `cannot open ../../include/sha256_gpu.hpp` | Make sure you `%cd gpu_assignment` before compiling |
| `libssl not found` | Re-run Step 3 (`apt-get install libssl-dev`) |
| `CUDA error: out of memory` | Reduce N in Step 9 — Colab T4 has 16 GB; 10M messages needs ~320 MB for digests |
| Dataset check `MISMATCH` | Re-run Step 5 to regenerate the dataset, then retry the validator |